# Summarize power-flow statistics across weather years

## Import packages

In [ ]:
## Time packages
import time
from datetime import datetime

## Memory packages
import psutil # tracking memory and cpu usage
import resource  # tracking memory and cpu usage
import gc
import sys

## Data structure packages
import numpy as np 
import pandas as pd # to create data frames
import pyarrow as pa

from collections import defaultdict

import re

## Math and logic packages
import random
import math

## Load & Save data packages
import os
import glob
import json
import yaml
import joblib

## Plotting packages
import matplotlib.pyplot as plt

## Packages for merging dictionaries
import os
from typing import List, Dict, Any, Iterable, Optional, Tuple, Union
import pandas as pd
import warnings
from joblib import load as joblib_load, dump as joblib_dump

import matplotlib.cm as cm

## Internal functions packages
from src import figure_ops
from src import input_ops
from src import df_ops
from src import file_ops

## Define functions

In [ ]:
def expand_years_from_TGW_ranges(config, scenario_name):
    years = []

    for entry in config.get("TGW_years_scenarios_ranges", []):
        scenarios_i = entry.get("scenarios", [])

        if scenario_name not in scenarios_i:
            continue

        start_year = int(entry["start_year"])
        end_year = int(entry["end_year"])

        years.extend(range(start_year, end_year + 1))

    return sorted(years)

def _parse_annual_summary_column(col):
    """
    Parse annual summary-stat columns like:
        max_loading_historical_1990
        95th_loading_rcp45hotter_2030
        hours_over_100_loading_rcp45hotter_2030

    Returns:
        (base_metric, scenario, year)

    Examples:
        max_loading_historical_1990
            -> ("max_loading", "historical", 1990)

        hours_over_100_loading_rcp45hotter_2030
            -> ("hours_over_100_loading", "rcp45hotter", 2030)

    Static columns and diff columns return None.
    """

    # Drop all annual pairwise-difference columns completely
    if "_diff_" in col or "_vs_historical_" in col:
        return None

    m = re.match(
        r"^(?P<base_metric>.+)_(?P<scenario>historical|[A-Za-z0-9]+)_(?P<year>\d{4})$",
        col,
    )

    if m is None:
        return None

    return (
        m.group("base_metric"),
        m.group("scenario"),
        int(m.group("year")),
    )


def _period_label(years):
    years = sorted({int(y) for y in years})

    if len(years) == 0:
        return "unknown"

    if len(years) == 1:
        return str(years[0])

    return f"{years[0]}_{years[-1]}"


def _infer_asset_id_cols(df):
    """
    Infer the unique asset column.

    For transformers:
        Transformer

    For lines:
        Line
    """

    if "Transformer" in df.columns:
        return ["Transformer"]

    if "Line" in df.columns:
        return ["Line"]

    raise ValueError(
        "Could not infer asset ID column. "
        "Pass asset_id_cols explicitly, e.g., ['Transformer'] or ['Line']."
    )


def _make_indexed(df, asset_id_cols):
    out = df.copy()

    missing = [c for c in asset_id_cols if c not in out.columns]
    if missing:
        raise ValueError(f"Missing asset ID columns: {missing}")

    out = out.set_index(asset_id_cols, drop=False)

    if not out.index.is_unique:
        raise ValueError(
            "Asset index is not unique. Use a more specific asset_id_cols list, "
            "for example ['Transformer', 'Bus'] or ['Line', 'LineCode']."
        )

    return out


def _concat_year_series(year_to_series):
    """
    Convert:
        {1990: Series, 1991: Series, ...}

    into a wide dataframe with years as columns.
    """

    if not year_to_series:
        return None

    return pd.concat(
        {int(year): series for year, series in sorted(year_to_series.items())},
        axis=1,
    )


def _row_quantile(wide, q):
    return wide.apply(pd.to_numeric, errors="coerce").quantile(
        q,
        axis=1,
        interpolation="linear",
    )


def _row_sum(wide):
    return wide.apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1)


def _row_median(wide):
    return wide.apply(pd.to_numeric, errors="coerce").median(axis=1)


# ------------------------------------------------------------
# Main function for one region
# ------------------------------------------------------------

def summarize_one_region_across_weather_years(
    merged_dict_weather,
    pair_keys,
    region_key,
    asset_id_cols=None,
):
    """
    Summarize annual asset-level summary statistics across multiple weather-year pairs
    for one region.

    Input:
        merged_dict_weather[(hist_year, fut_year, fut_scenario)][region_key] = df

    Output:
        One dataframe with static asset metadata plus period-level summary columns.
    """

    metric_series = defaultdict(dict)
    static_frames = []
    first_df = None

    for weather_pair_key in sorted(pair_keys, key=lambda x: (int(x[0]), int(x[1]), x[2])):

        if region_key not in merged_dict_weather.get(weather_pair_key, {}):
            continue

        df = merged_dict_weather[weather_pair_key][region_key].copy()

        if first_df is None:
            first_df = df.copy()

            if asset_id_cols is None:
                asset_id_cols = _infer_asset_id_cols(first_df)

        df_idx = _make_indexed(df, asset_id_cols)

        # Static columns = non-annual columns, excluding diff columns
        static_cols = [
            c for c in df.columns
            if _parse_annual_summary_column(c) is None
            and "_diff_" not in c
            and "_vs_historical_" not in c
        ]

        static_frames.append(df_idx[static_cols])

        # Collect annual metric columns by scenario, metric name, and year
        for col in df.columns:
            parsed = _parse_annual_summary_column(col)

            if parsed is None:
                continue

            base_metric, scenario, year = parsed

            metric_series[(scenario, base_metric)][year] = pd.to_numeric(
                df_idx[col],
                errors="coerce",
            )

    if first_df is None:
        raise ValueError(f"No dataframe found for region_key={region_key}")

    static_df = (
        pd.concat(static_frames, axis=0)
        .groupby(level=list(range(len(asset_id_cols))), sort=False)
        .first()
    )

    result_series = []

    scenarios = sorted({scenario for scenario, _base_metric in metric_series.keys()})

    for scenario in scenarios:

        scenario_years = sorted({
            year
            for (scenario_i, _base_metric), year_map in metric_series.items()
            if scenario_i == scenario
            for year in year_map.keys()
        })

        scen_period = _period_label(scenario_years)
        suffix = f"{scenario}_{scen_period}"

        base_metrics = sorted({
            base_metric
            for (scenario_i, base_metric) in metric_series.keys()
            if scenario_i == scenario
        })

        # ------------------------------------------------------------
        # Summarize non-share metrics
        # ------------------------------------------------------------

        for base_metric in base_metrics:

            # share_hours_* is recomputed later from summed hours / summed n_hours
            if base_metric.startswith("share_hours_over_"):
                continue

            wide = _concat_year_series(metric_series[(scenario, base_metric)])

            if wide is None:
                continue

            # 1. Min values:
            #    min, p5, median across annual values
            if base_metric == "min_loading":

                result_series.append(
                    wide.min(axis=1, skipna=True).rename(
                        f"min_annual_{base_metric}_{suffix}"
                    )
                )

                result_series.append(
                    _row_quantile(wide, 0.05).rename(
                        f"p5_annual_{base_metric}_{suffix}"
                    )
                )

                result_series.append(
                    _row_median(wide).rename(
                        f"median_annual_{base_metric}_{suffix}"
                    )
                )

            # 2. Max values:
            #    median, p95, max across annual values
            elif base_metric == "max_loading":

                result_series.append(
                    _row_median(wide).rename(
                        f"median_annual_{base_metric}_{suffix}"
                    )
                )

                result_series.append(
                    _row_quantile(wide, 0.95).rename(
                        f"p95_annual_{base_metric}_{suffix}"
                    )
                )

                result_series.append(
                    wide.max(axis=1, skipna=True).rename(
                        f"max_annual_{base_metric}_{suffix}"
                    )
                )

            # 3. Percentile values:
            #    median and max across annual percentiles
            #
            # Examples:
            #    90th_loading
            #    95th_loading
            #    99th_loading
            elif re.match(r"^\d+(st|nd|rd|th)_loading$", base_metric):

                result_series.append(
                    _row_median(wide).rename(
                        f"median_annual_{base_metric}_{suffix}"
                    )
                )

                result_series.append(
                    wide.max(axis=1, skipna=True).rename(
                        f"max_annual_{base_metric}_{suffix}"
                    )
                )

            # 4. n_hours, n_unique_hours:
            #    sum across years
            elif base_metric in {"n_hours", "n_unique_hours"}:

                result_series.append(
                    _row_sum(wide).rename(
                        f"total_{base_metric}_{suffix}"
                    )
                )

            # 5. hours_over_*:
            #    sum and median across annual values
            elif re.match(r"^hours_over_\d+_loading$", base_metric):

                result_series.append(
                    _row_sum(wide).rename(
                        f"total_{base_metric}_{suffix}"
                    )
                )

                result_series.append(
                    _row_median(wide).rename(
                        f"median_annual_{base_metric}_{suffix}"
                    )
                )

            elif base_metric == "median_loading":

                result_series.append(
                    _row_median(wide).rename(
                        f"median_annual_{base_metric}_{suffix}"
                    )
                )

            elif base_metric == "average_loading":

                # Weighted average across all simulated hours:
                # sum(annual_average * annual_n_hours) / sum(annual_n_hours)
                n_hours_wide = _concat_year_series(
                    metric_series.get((scenario, "n_hours"), {})
                )

                if n_hours_wide is not None:
                    common_years = sorted(
                        set(wide.columns).intersection(set(n_hours_wide.columns))
                    )

                    if common_years:
                        numerator = (
                            wide[common_years].astype(float)
                            * n_hours_wide[common_years].astype(float)
                        ).sum(axis=1, min_count=1)

                        denominator = (
                            n_hours_wide[common_years]
                            .astype(float)
                            .sum(axis=1, min_count=1)
                        )

                        weighted_average = numerator / denominator.replace(0, np.nan)

                        result_series.append(
                            weighted_average.rename(
                                f"weighted_average_{base_metric}_{suffix}"
                            )
                        )

                result_series.append(
                    _row_median(wide).rename(
                        f"median_annual_{base_metric}_{suffix}"
                    )
                )

            else:
                result_series.append(
                    _row_median(wide).rename(
                        f"median_annual_{base_metric}_{suffix}"
                    )
                )

                result_series.append(
                    wide.max(axis=1, skipna=True).rename(
                        f"max_annual_{base_metric}_{suffix}"
                    )
                )

        # ------------------------------------------------------------
        # 6. share_hours_*:
        #    recompute from summed hours_over_* / summed n_hours
        # ------------------------------------------------------------

        n_hours_wide = _concat_year_series(
            metric_series.get((scenario, "n_hours"), {})
        )

        if n_hours_wide is not None:

            total_n_hours = _row_sum(n_hours_wide)

            share_bases = sorted({
                base_metric
                for (scenario_i, base_metric) in metric_series.keys()
                if scenario_i == scenario
                and base_metric.startswith("share_hours_over_")
            })

            for share_base in share_bases:

                hours_base = share_base.replace("share_", "", 1)

                hours_wide = _concat_year_series(
                    metric_series.get((scenario, hours_base), {})
                )

                if hours_wide is None:
                    continue

                total_hours_over = _row_sum(hours_wide)

                share = total_hours_over / total_n_hours.replace(0, np.nan)

                result_series.append(
                    share.rename(
                        f"{share_base}_{suffix}"
                    )
                )

    summary_df = pd.concat([static_df] + result_series, axis=1).reset_index(drop=True)

    return summary_df


# ------------------------------------------------------------
# Main function for full nested dictionary
# ------------------------------------------------------------

def summarize_summary_dict_across_weather_years(
    merged_dict_weather,
    asset_id_cols=None,
):
    """
    Create a period-level summary dictionary from a merged weather-pair dictionary.

    Input structure:
        merged_dict_weather[
            (historical_year, future_year, future_scenario)
        ][
            (smart_ds_year, city, region)
        ] = annual_pair_df

    Output structure:
        summary_dict[
            (future_period, future_scenario)
        ][
            (smart_ds_year, city, region)
        ] = period_summary_df

    Example output level-1 key:
        ('2030_2059', 'rcp45hotter')
    """

    if not merged_dict_weather:
        raise ValueError("merged_dict_weather is empty")

    pair_keys_by_future_scenario = defaultdict(list)

    for key in merged_dict_weather.keys():

        if len(key) != 3:
            raise ValueError(
                "Expected level-1 keys of the form "
                "(historical_year, future_year, future_scenario). "
                f"Found: {key}"
            )

        hist_year, fut_year, fut_scenario = key
        pair_keys_by_future_scenario[fut_scenario].append(key)

    summary_dict = {}

    for fut_scenario, pair_keys in pair_keys_by_future_scenario.items():

        hist_period = _period_label([k[0] for k in pair_keys])
        fut_period = _period_label([k[1] for k in pair_keys])

        out_key = (fut_period, fut_scenario)

        region_keys = sorted(
            {
                region_key
                for key in pair_keys
                for region_key in merged_dict_weather.get(key, {}).keys()
            },
            key=str,
        )

        summary_dict[out_key] = {}

        for region_key in region_keys:

            summary_dict[out_key][region_key] = summarize_one_region_across_weather_years(
                merged_dict_weather=merged_dict_weather,
                pair_keys=pair_keys,
                region_key=region_key,
                asset_id_cols=asset_id_cols,
            )

    return summary_dict


## Load config file with scenarios and parameters 

In [ ]:
config_file_name = 'opendss_config1'; config_path = f"config/{config_file_name}.yaml"; config = input_ops.load_config(config_path)

CITY_REGIONS_TO_RUN = config['CITY_REGIONS_TO_RUN']

smart_ds_year = config['smart_ds_years'][0]

## Initialize parameters for saving paths
output_pf_path = config['output_pf_path']
    
# Percent-of-peak range to load (e.g., top 0–10% hours)
start_row_percent = config['start_row_percent']
top_percent_mdh = config['top_percent_mdh']

# File names
transformers_file_name = f"transformers_top_{start_row_percent}_{top_percent_mdh}_percent"
lines_file_name = f"lines_top_{start_row_percent}_{top_percent_mdh}_percent"

top_n_hours = int(np.ceil(8760*top_percent_mdh/100)) # calculate top city demand hours to run (top_percent_mdh% of hours of the year)

TGW_years_scenarios_ranges = config.get("TGW_years_scenarios_ranges", [])

# Set TGW future scenario
comparison_TGW_scenario = "rcp45hotter"

# Solar / battery SMART-DS scenario parameters
solar_share = config.get("solar_share", "none")
battery_share = config.get("battery_share", "none")

solar_battery_scenario_folder = input_ops.build_solar_battery_scenario_folder(
    solar_share=solar_share,
    battery_share=battery_share,
)

# Count the total number of regions across all cities
num_regions = sum(len(regions) for regions in CITY_REGIONS_TO_RUN.values())

save_folder = "all_regions"

print(f"solar_share: {solar_share}\n battery_share: {battery_share}\n solar_battery_scenario_folder: {solar_battery_scenario_folder}")


print(f"\nsmart_ds_year: {smart_ds_year}  \n\nstart_row_percent: {start_row_percent} \n\ntop_percent_mdh: {top_percent_mdh} \n\nOutput_pf_path: {output_pf_path}")

print(f"\nTGW_scenario: {comparison_TGW_scenario}")


print(f"\nfor weather year comparison: {TGW_years_scenarios_ranges}")

print(f"\nCITY_REGIONS_TO_RUN = {CITY_REGIONS_TO_RUN}")

print(f"\nRunning {num_regions} regions; save_folder = {save_folder!r}")


## Load dictionaries for multiple weather-year pairs (single RCP)

In [ ]:
# ============================================================
# Load dictionaries for multiple weather year pairs
# Single solar-battery scenario, multiple paired weather years
# ============================================================

# solar-battery scenario 
solar_share = config.get("solar_share", "none")
battery_share = config.get("battery_share", "none")

solar_battery_scenario_folder = input_ops.build_solar_battery_scenario_folder(
    solar_share=solar_share,
    battery_share=battery_share,
)

print(
    "Solar-battery scenario:\n"
    f"  solar_share: {solar_share}\n"
    f"  battery_share: {battery_share}\n"
    f"  folder: {solar_battery_scenario_folder}"
)

smart_ds_year = config["smart_ds_years"][0]

cities = ["AUS", "GSO", "SFO"]

CITY_MAP = {
    "AUS": "Austin",
    "GSO": "Greensboro",
    "SFO": "San-Francisco",
}

# Grid reinforcement thresholds
xfm_cand = 80
xfm_crit = 100
line_cand = 67
line_crit = 100


# ------------------------------------------------------------
# Build weather-year pairs from TGW_years_scenarios_ranges
# Example:
#   1990 historical -> 2030 rcp45hotter
#   ...
#   2019 historical -> 2059 rcp45hotter
# ------------------------------------------------------------


historical_years = expand_years_from_TGW_ranges(
    config,
    scenario_name="historical",
)

future_years = expand_years_from_TGW_ranges(
    config,
    scenario_name=comparison_TGW_scenario,
)

if len(historical_years) != len(future_years):
    raise ValueError(
        "Historical and future year ranges must have the same number of years. "
        f"Found {len(historical_years)} historical years and {len(future_years)} future years."
    )

weather_year_pairs = list(zip(historical_years, future_years))

# Validate 40-year pairing
bad_pairs = [
    (hist_year, fut_year)
    for hist_year, fut_year in weather_year_pairs
    if fut_year - hist_year != 40
]

if bad_pairs:
    raise ValueError(
        "All historical–future weather-year pairs are expected to have a 40-year difference. "
        f"Pairs with unexpected offsets: {bad_pairs}"
    )


weather_pair_keys = [
    (str(hist_year), str(fut_year), comparison_TGW_scenario)
    for hist_year, fut_year in weather_year_pairs
]

print("\nWeather-year pairs to compare:")
for hist_year, fut_year, fut_scenario in weather_pair_keys:
    print(f"  {hist_year} historical vs {fut_year} {fut_scenario}")


merged_transformers_dict_weather = {}
merged_lines_dict_weather = {}

missing_files = []

for hist_year, fut_year, fut_scenario in weather_pair_keys:

    weather_pair_key = (hist_year, fut_year, fut_scenario)
    future_climate_key = (fut_year, fut_scenario)

    merged_transformers_dict_weather.setdefault(weather_pair_key, {})
    merged_lines_dict_weather.setdefault(weather_pair_key, {})

    for city, regions in CITY_REGIONS_TO_RUN.items():
        for region in regions:

            region_key = (smart_ds_year, city, region)

            save_dir = os.path.join(
                output_pf_path,
                city,
                region,
                fut_scenario,
                fut_year,
                solar_battery_scenario_folder,
            )

            xfer_summary_save_path = os.path.join(
                save_dir,
                f"{transformers_file_name}_summary.joblib",
            )

            lines_summary_save_path = os.path.join(
                save_dir,
                f"{lines_file_name}_summary.joblib",
            )
            
            if not os.path.exists(xfer_summary_save_path):
                missing_files.append(xfer_summary_save_path)
                continue

            if not os.path.exists(lines_summary_save_path):
                missing_files.append(lines_summary_save_path)
                continue
                
            transformers_dict_loaded = joblib.load(xfer_summary_save_path)
            lines_dict_loaded = joblib.load(lines_summary_save_path)

            df_xfm = transformers_dict_loaded[future_climate_key][region_key].copy()
            df_line = lines_dict_loaded[future_climate_key][region_key].copy()

            merged_transformers_dict_weather[weather_pair_key][region_key] = df_xfm
            merged_lines_dict_weather[weather_pair_key][region_key] = df_line


if missing_files:
    print("\nWARNING: Missing files:")
    for path in missing_files:
        print(path)

print("\nDictionary nested keys structure:")
file_ops.print_nested_keys_structure(merged_transformers_dict_weather)

print("\nDictionary sample keys and dataframe structure:")
file_ops.print_nested_dict_key_examples_and_dataframe_details(
    merged_transformers_dict_weather
)

## Create dictionary with summary of summary statistics

In [ ]:
# ============================================================
# Summarize annual summary-stat dictionaries across weather years
# ============================================================

start_time = time.time()


summary_transformers_dict_weather = summarize_summary_dict_across_weather_years(
    merged_transformers_dict_weather
)

summary_lines_dict_weather = summarize_summary_dict_across_weather_years(
    merged_lines_dict_weather
)


print("\nTransformer summary dictionary nested keys:")
file_ops.print_nested_keys_structure(summary_transformers_dict_weather)

print("\nTransformer summary dictionary sample dataframe:")
file_ops.print_nested_dict_key_examples_and_dataframe_details(
    summary_transformers_dict_weather
)


print("\nLines summary dictionary nested keys:")
file_ops.print_nested_keys_structure(summary_lines_dict_weather)

print("\nLines summary dictionary sample dataframe:")
file_ops.print_nested_dict_key_examples_and_dataframe_details(
    summary_lines_dict_weather
)

end_time = time.time(); print("Runtime:", (end_time - start_time) / 60, "minutes")


## Save

In [ ]:
# ============================================================
# save period-level summary dictionaries
# ============================================================
summary_save_dir = os.path.join(
    output_pf_path,
    save_folder,
    "summary_across_weather_years",
    comparison_TGW_scenario,
    solar_battery_scenario_folder,
)

os.makedirs(summary_save_dir, exist_ok=True)

transformers_summary_across_years_path = os.path.join(
    summary_save_dir,
    f"{transformers_file_name}_summary_across_weather_years.joblib",
)

lines_summary_across_years_path = os.path.join(
    summary_save_dir,
    f"{lines_file_name}_summary_across_weather_years.joblib",
)

joblib.dump(
    summary_transformers_dict_weather,
    transformers_summary_across_years_path,
)

joblib.dump(
    summary_lines_dict_weather,
    lines_summary_across_years_path,
)

print("Saved:")
print(transformers_summary_across_years_path)
print(lines_summary_across_years_path)